# Sensor-cal validation plotter

Compares **calibrated** (post `sensor_cal_hw`, `sensor_cal_s`) image-side measurements against Gazebo ground-truth derived quantities. Modeled after `~/ws/scripts/soft_precise_landing/plotter_calibration.ipynb` cells 11+16+18 but pointed at the PX4_Gazebo recording layout.

Defaults to the `calibration_data/latest` symlink. Override `RUN_DIR` if you want to inspect a specific timestamped run.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter as sgf
from ahrs import Quaternion, DCM

np.set_printoptions(precision=3, suppress=True)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# Pick a run directory. Defaults to the most recent timestamped folder.
# Set RUN_DIR explicitly to inspect a specific recording, e.g.
#   RUN_DIR = os.path.join(PARENT, 'Tue May 12 15-26-29 2026')
PARENT = '/home/shubham/Soft-Precise-Landing/PX4_Gazebo/calibration_data/output'
_cands = [d for d in os.listdir(PARENT) if os.path.isdir(os.path.join(PARENT, d))]
RUN_DIR = os.path.join(PARENT, max(_cands, key=lambda d: os.path.getmtime(os.path.join(PARENT, d))))
print(f'Loading from: {RUN_DIR}')

img  = np.load(f'{RUN_DIR}/Img_Data.npy',       allow_pickle=True)[()]
tel  = np.load(f'{RUN_DIR}/Telemetry_Data.npy', allow_pickle=True)[()]
gt   = np.load(f'{RUN_DIR}/Ground_Truth.npy',   allow_pickle=True)[()]

print('img keys:', list(img.keys()))
print('tel keys:', list(tel.keys()))
print('gt  keys:', list(gt.keys()))
print(f'samples: img={len(img["Time"])}, gt={len(gt["Time"])}')

## Currently-applied sensor_cal matrices

Edit these to match what's in `img_data.py` if you want to validate a specific candidate set.

In [ ]:
# Set these to whatever you want to evaluate.
# Defaults reflect the 2026-05-13 nanmedian across 4 fresh runs (after lstsq + LK fix).
sensor_cal_hw = np.diag([0.1498, 0.1694, 0.0877, 0.2188, 0.2114, 0.4236])
sensor_cal_s  = np.diag([0.6069, 0.6109, 1.0,    1.0])

print('sensor_cal_hw =\n', sensor_cal_hw)
print('sensor_cal_s  =\n', sensor_cal_s)

## Ground truth from UAV/target poses (Gazebo)

Port of plotter_calibration cells 16 + 18:
- **W** = Gazebo world frame (ENU)
- **B** = body FRD (forward / right / down)
- `FLU_2_FRD = DCM(x=180°)` converts the rotation-matrix-derived FLU body frame into FRD

In [ ]:
FLU_2_FRD = np.array(DCM(x=180.0))

t_g_all = np.array(gt['Time'])
dt = np.diff(t_g_all)
valid = np.hstack(([True], dt > 1e-6))
t_g = t_g_all[valid]
n = len(t_g)
print(f'{n} valid samples over {t_g[-1]-t_g[0]:.1f}s')

uav_poses    = np.array(gt['UAV Pose'],    dtype=object)[valid]
target_poses = np.array(gt['Target Pose'], dtype=object)[valid]

W_T_P  = np.zeros((n, 4, 4))
W_R_T  = np.zeros((n, 3, 3))
W_x_tu = np.zeros((n, 3))
B_x_tu = np.zeros((n, 3))
for i, (p, tp) in enumerate(zip(uav_poses, target_poses)):
    Ru = Quaternion([p.orientation.w, p.orientation.x, p.orientation.y, p.orientation.z]).to_DCM()
    Rt = Quaternion([tp.orientation.w, tp.orientation.x, tp.orientation.y, tp.orientation.z]).to_DCM()
    W_T_P[i, :3, :3] = Ru
    W_T_P[i, :3, 3]  = [p.position.x, p.position.y, p.position.z]
    W_T_P[i, 3, 3]   = 1.0
    W_R_T[i]         = Rt
    W_x_t            = np.array([tp.position.x, tp.position.y, tp.position.z])
    W_x_tu[i]        = W_x_t - W_T_P[i, :3, 3]
    B_x_tu[i]        = FLU_2_FRD @ np.linalg.inv(Ru) @ W_x_tu[i]

# Body-frame target velocity
W_x_tu_filt = sgf(W_x_tu, 51, 2, axis=0)
W_v_tu      = np.gradient(W_x_tu_filt, t_g, axis=0)
B_v_tu      = np.zeros((n, 3))
for i in range(n):
    B_v_tu[i] = FLU_2_FRD @ np.linalg.inv(W_T_P[i, :3, :3]) @ W_v_tu[i]

# Ground-truth optical flow = body-frame target velocity / depth
z = B_x_tu[:, 2].copy()
z[np.abs(z) < 0.1] = np.nan
B_y_g = B_v_tu / z[:, np.newaxis]

# Ground-truth angular velocity (target wrt UAV in body frame)
W_dR_B = np.gradient(W_T_P[:, :3, :3], t_g, axis=0)
B_w_ug = np.zeros((n, 3))
for i in range(n):
    skew = W_T_P[i, :3, :3].T @ W_dR_B[i]
    B_w_ug[i] = FLU_2_FRD @ np.array([skew[2, 1], skew[0, 2], skew[1, 0]])
W_dR_T = np.gradient(W_R_T, t_g, axis=0)
B_w_tg = np.zeros((n, 3))
for i in range(n):
    skew = W_R_T[i].T @ W_dR_T[i]
    B_w_tg[i] = FLU_2_FRD @ np.array([skew[2, 1], skew[0, 2], skew[1, 0]])
B_w_tug = B_w_tg - B_w_ug

# Ground-truth centroid (xc, yc)
xc_gt = B_x_tu[:, 0] / B_x_tu[:, 2]
yc_gt = B_x_tu[:, 1] / B_x_tu[:, 2]

## Image-side measurements (calibrated)

We logged the RAW values via `getRawOptFlowAngVel` / `getRawImgFeatureParam` from `img_data.py`. Apply the candidate `sensor_cal` matrices to see how the calibrated outputs compare to ground truth.

In [ ]:
# Apply Savitzky-Golay filter on the raw image-side measurements before sensor_cal.
# Parameters retuned 2026-05-12 via tune_savgol.py: sweep over windows
# [5..101] × polyorders [1..4] across all 5 calibration recordings; selected
# (window=101, polyorder=3) for maximum mean|corr|. Improves over MATLAB's
# default (11, 2) by +26% mean correlation (0.348 → 0.439).
#
# NOTE: window=101 at ~30 Hz = ~3.37 s of group delay. Fine for offline
# analysis (this notebook), NOT suitable for the live PLASMC controller —
# img_data.py runtime should use a much shorter window (e.g. MATLAB's 11)
# or no filter at all.
FILTER_WIN = 101     # was 11 (MATLAB Constants.m), retuned 2026-05-12
POLYORDER  = 3       # was 2 (MATLAB sgolayfilt order), retuned 2026-05-12

raw_hw = np.asarray(gt['Opt Flow Ang Vel'])[valid]   # (N, 6) raw lstsq output
raw_s  = np.asarray(gt['Img Feature Params'])[valid] # (N, 4) raw centroid/alpha

if len(raw_hw) >= FILTER_WIN:
    raw_hw_filt = sgf(raw_hw, FILTER_WIN, POLYORDER, axis=0)
    raw_s_filt  = sgf(raw_s,  FILTER_WIN, POLYORDER, axis=0)
else:
    raw_hw_filt = raw_hw.copy()
    raw_s_filt  = raw_s.copy()

cal_hw = (sensor_cal_hw @ raw_hw_filt.T).T
cal_s  = (sensor_cal_s  @ raw_s_filt.T ).T
y_cal, w_cal     = cal_hw[:, :3], cal_hw[:, 3:]
xc_cal, yc_cal   = cal_s[:, 0], cal_s[:, 1]

# unsmoothed for side-by-side comparison
cal_hw_unsmoothed = (sensor_cal_hw @ raw_hw.T).T
cal_s_unsmoothed  = (sensor_cal_s  @ raw_s.T ).T
y_cal_raw, w_cal_raw = cal_hw_unsmoothed[:, :3], cal_hw_unsmoothed[:, 3:]
xc_cal_raw, yc_cal_raw = cal_s_unsmoothed[:, 0], cal_s_unsmoothed[:, 1]

print(f'savgol applied: window={FILTER_WIN}, polyorder={POLYORDER}')
print(f'   (offline-tuned for max |corr|; live controller should keep window<=11)\n')
print('calibrated optical flow RMS    (smoothed):', np.sqrt(np.nanmean(y_cal**2, axis=0)))
print('calibrated optical flow RMS  (unsmoothed):', np.sqrt(np.nanmean(y_cal_raw**2, axis=0)))
print('ground truth   optical flow RMS:           ', np.sqrt(np.nanmean(B_y_g**2, axis=0)))
print()
print('calibrated ang vel RMS    (smoothed):', np.sqrt(np.nanmean(w_cal**2, axis=0)))
print('calibrated ang vel RMS  (unsmoothed):', np.sqrt(np.nanmean(w_cal_raw**2, axis=0)))
print('ground truth ang vel RMS:             ', np.sqrt(np.nanmean(B_w_tug**2, axis=0)))
print()
print(f'calibrated centroid RMS (smoothed):   xc={np.sqrt(np.nanmean(xc_cal**2)):.4f}  yc={np.sqrt(np.nanmean(yc_cal**2)):.4f}')
print(f'ground truth centroid RMS:           xc={np.sqrt(np.nanmean(xc_gt **2)):.4f}  yc={np.sqrt(np.nanmean(yc_gt **2)):.4f}')

## Plots — calibrated vs ground truth

In [ ]:
def overlay_axes(ax, t, gt_v, cal_v, title, ylabel, ylim=None):
    ax.plot(t, gt_v,  label='Ground truth (Gazebo pose)', linewidth=2, alpha=0.85)
    ax.plot(t, cal_v, label='Calibrated (image-side)',     linewidth=1.2, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('t (s)')
    ax.set_ylabel(ylabel)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend(loc='upper right', fontsize=9)

### Optical flow (3 axes)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, ax in enumerate(axes):
    overlay_axes(ax, t_g, B_y_g[:, i], y_cal[:, i],
                 title=f'Optical flow axis {i}', ylabel=f'y_{i} (rad/s)')
plt.show()

### Angular velocity (3 axes)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, ax in enumerate(axes):
    overlay_axes(ax, t_g, B_w_tug[:, i], w_cal[:, i],
                 title=f'Angular velocity axis {i}', ylabel=f'w_{i} (rad/s)')
plt.show()

### Centroid (xc, yc)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)
overlay_axes(axes[0], t_g, xc_gt, xc_cal, title='Centroid x (normalized)', ylabel='xc')
overlay_axes(axes[1], t_g, yc_gt, yc_cal, title='Centroid y (normalized)', ylabel='yc')
plt.show()

### UAV trajectory (sanity-check the recorded motion)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
labels = ['x (m, world)', 'y (m, world)', 'z (m, world)']
for i, (ax, lab) in enumerate(zip(axes, labels)):
    ax.plot(t_g, W_T_P[:, i, 3], label='UAV')
    ax.plot(t_g, np.array([t.position.x if i==0 else (t.position.y if i==1 else t.position.z) for t in target_poses]), label='Target', linestyle='--')
    ax.set_title(f'World {lab}')
    ax.set_xlabel('t (s)')
    ax.set_ylabel(lab)
    ax.legend(loc='upper right')
plt.show()

## Calibration quality metrics

Per-axis Pearson correlation and RMS error (calibrated vs ground truth). Higher correlation + lower RMS error = better calibration.

In [ ]:
def quality(name, gt_v, cal_v):
    mask = np.isfinite(gt_v) & np.isfinite(cal_v)
    if mask.sum() < 10:
        return float('nan'), float('nan')
    g, c = gt_v[mask], cal_v[mask]
    corr = np.corrcoef(g, c)[0, 1]
    rmse = np.sqrt(np.mean((g - c) ** 2))
    return corr, rmse

def report(name, gt_v, cal_v_smooth, cal_v_raw):
    c_s, e_s = quality(name, gt_v, cal_v_smooth)
    c_r, e_r = quality(name, gt_v, cal_v_raw)
    print(f'  {name:6s}  smoothed: corr={c_s:+.3f} RMSE={e_s:.4f}'
          f'   unsmoothed: corr={c_r:+.3f} RMSE={e_r:.4f}')

print('Optical flow axes (smoothed vs unsmoothed):')
for i in range(3):
    report(f'y_{i}', B_y_g[:, i], y_cal[:, i], y_cal_raw[:, i])
print()
print('Angular velocity axes (smoothed vs unsmoothed):')
for i in range(3):
    report(f'w_{i}', B_w_tug[:, i], w_cal[:, i], w_cal_raw[:, i])
print()
print('Centroid axes (smoothed vs unsmoothed):')
report('xc', xc_gt, xc_cal, xc_cal_raw)
report('yc', yc_gt, yc_cal, yc_cal_raw)

## Online filter A/B: KF vs Savgol vs ground truth

The runtime `img_data.py` now computes and logs **both** filter outputs every frame:
- `Opt Flow KF`     — per-channel 2-state constant-velocity Kalman filter (causal, adaptive gain)
- `Opt Flow Savgol` — legacy Savgol(13, 1) sliding-window filter (non-causal, fixed lag)

Both are sampled at image rate and saved into `Img_Data.npy`. The active controller filter
is chosen by the `IMG_FILTER` env var (`kf` default, `savgol` for the legacy path), but
both are logged regardless so a single calibration run gives an apples-to-apples comparison.

The cells below load both online filter streams, align them to the GT time origin via
`gt['Start Time']`, interpolate ground truth onto image timestamps, and report RMS-error
vs GT plus HF-noise content per channel.


In [ ]:
# --- Load online filter outputs + align time axis ---
kf_log = np.asarray(img.get('Opt Flow KF', []))
sg_log = np.asarray(img.get('Opt Flow Savgol', []))
if len(kf_log) == 0 or len(sg_log) == 0:
    raise RuntimeError(
        "This run was recorded with an older img_data.py — no 'Opt Flow KF' / "
        "'Opt Flow Savgol' fields in Img_Data.npy. Re-record a calibration run "
        "with the current code to use the online A/B analysis below."
    )

# Image timestamps in the same origin as gt time (t_g)
img_t_abs = np.asarray(img['Time'])
img_t_rel = img_t_abs - gt['Start Time']

# The filter logs were appended only on FEATURE_DATA_IS_LOGGED frames (valid LK);
# the raw buffer additionally has zeros on LK-fail frames. Build a mask aligning
# the filter-log indices with img['Time'].
raw_im   = np.asarray(img['Opt Flow Ang Vel'])
valid_im = (raw_im != 0).any(axis=1)
n_pair   = min(int(valid_im.sum()), len(kf_log), len(sg_log))
t_valid  = img_t_rel[valid_im][:n_pair]
kf_log   = kf_log[:n_pair]
sg_log   = sg_log[:n_pair]

# Trim to GT time window
t_lo, t_hi = max(t_valid.min(), t_g.min()), min(t_valid.max(), t_g.max())
mask_im = (t_valid >= t_lo) & (t_valid <= t_hi)
t_valid = t_valid[mask_im]; kf_log = kf_log[mask_im]; sg_log = sg_log[mask_im]
mask_gt = (t_g >= t_lo) & (t_g <= t_hi)
t_g_w   = t_g[mask_gt]
B_y_g_w = B_y_g[mask_gt]
B_w_tug_w = B_w_tug[mask_gt]

# Interpolate GT (6-vec) onto img time grid for direct comparison
gt6 = np.hstack([B_y_g_w, B_w_tug_w])
gt6_im = np.zeros((len(t_valid), 6))
for i in range(6):
    gt6_im[:, i] = np.interp(t_valid, t_g_w, gt6[:, i])

print(f'aligned: n_pair={len(t_valid)}, t=[{t_valid[0]:.2f}, {t_valid[-1]:.2f}] s')
print(f'GT samples in window: {len(t_g_w)}')


### Optical flow + angular velocity (6 channels): KF, Savgol, GT

In [ ]:
CHN = ['flow_x', 'flow_y', 'flow_z', 'ω_x', 'ω_y', 'ω_z']
fig, axes = plt.subplots(6, 1, figsize=(12, 14), sharex=True, constrained_layout=True)
for i, name in enumerate(CHN):
    ax = axes[i]
    ax.plot(t_g_w,   gt6[:, i],  color='k',  ls='--', lw=1.4, alpha=0.85, label='Ground truth')
    ax.plot(t_valid, sg_log[:, i], color='C1', lw=1.0, alpha=0.9, label='Savgol(13,1)')
    ax.plot(t_valid, kf_log[:, i], color='C0', lw=1.0, alpha=0.9, label='KF (q=5, r=0.1)')
    ax.set_ylabel(name)
    if i == 0: ax.legend(loc='upper right', fontsize=9)
axes[-1].set_xlabel('t (s, relative to sweep start)')
fig.suptitle('Online KF vs Savgol vs ground truth — full sweep', fontsize=12)
plt.show()


### Zoom: 5 s mid-run window (HF differences most visible)

In [ ]:
z_center = (t_lo + t_hi) / 2
zlo, zhi = z_center - 2.5, z_center + 2.5
zm_g = (t_g_w >= zlo) & (t_g_w <= zhi)
zm_i = (t_valid >= zlo) & (t_valid <= zhi)

fig, axes = plt.subplots(6, 1, figsize=(12, 14), sharex=True, constrained_layout=True)
for i, name in enumerate(CHN):
    ax = axes[i]
    ax.plot(t_g_w[zm_g],   gt6[zm_g, i],   color='k',  ls='--', lw=1.6, alpha=0.85, label='GT')
    ax.plot(t_valid[zm_i], sg_log[zm_i, i], color='C1', lw=1.4, alpha=0.9, label='Savgol')
    ax.plot(t_valid[zm_i], kf_log[zm_i, i], color='C0', lw=1.4, alpha=0.9, label='KF')
    ax.set_ylabel(name)
    if i == 0: ax.legend(loc='upper right', fontsize=9)
axes[-1].set_xlabel('t (s)')
fig.suptitle(f'Zoom: {zlo:.1f}–{zhi:.1f}s', fontsize=12)
plt.show()


### Quality metrics: RMS error vs GT + high-frequency content per channel

- **RMSE_vs_GT** — accuracy. Lower is better. Both filters should be similar; small differences indicate filter-specific tradeoffs.
- **HF_RMS** — high-frequency (1st-difference) RMS. Measures temporal jitter the filter passes through. Lower means smoother output.
- **HF reduction vs raw** — what fraction of the raw HF content the filter rejected.


In [ ]:
def rms(x):     return np.sqrt(np.mean(x**2, axis=0))
def hf_rms(x):  return np.sqrt(np.mean(np.diff(x, axis=0)**2, axis=0))

# Raw (calibrated) for HF-reduction context
raw_cal = raw_im[valid_im][:n_pair] @ sensor_cal_hw.T
raw_cal = raw_cal[mask_im]
raw_cal = raw_cal[zm_i.shape[0] != raw_cal.shape[0] and slice(None, len(t_valid)) or slice(None)] if False else raw_cal[:len(t_valid)]

err_kf = rms(kf_log - gt6_im)
err_sg = rms(sg_log - gt6_im)
hf_kf  = hf_rms(kf_log)
hf_sg  = hf_rms(sg_log)
hf_raw = hf_rms(raw_cal)

print('RMSE vs Ground Truth (per channel):')
print(f'  {"channel":8} {"KF":>10} {"Savgol":>10} {"Δ(KF-SG)":>10}')
for i, lbl in enumerate(CHN):
    delta = err_kf[i] - err_sg[i]
    flag  = '  ← KF better' if delta < -1e-3 else ('  ← Savgol better' if delta > 1e-3 else '')
    print(f'  {lbl:8} {err_kf[i]:10.4f} {err_sg[i]:10.4f} {delta:+10.4f}{flag}')

print('\nHigh-frequency (1st-diff) RMS per channel:')
print(f'  {"channel":8} {"raw":>10} {"KF":>10} {"Savgol":>10} {"KF/SG":>8}')
for i, lbl in enumerate(CHN):
    ratio = hf_kf[i] / hf_sg[i] if hf_sg[i] > 0 else float('nan')
    print(f'  {lbl:8} {hf_raw[i]:10.4f} {hf_kf[i]:10.4f} {hf_sg[i]:10.4f} {ratio:8.2f}')

print('\nHF reduction vs raw (%):')
print(f'  {"channel":8} {"KF":>10} {"Savgol":>10}')
for i, lbl in enumerate(CHN):
    print(f'  {lbl:8} {100*(1-hf_kf[i]/hf_raw[i]):10.1f} {100*(1-hf_sg[i]/hf_raw[i]):10.1f}')
